# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">10/7(수) 오전 · 보고서 형식 · 묶음 요약 · 위험도 정렬 — 실습</mark>

10/6 에는 LLM 에게 로그 한 줄씩을 읽혔습니다. 오늘은 **밤새 쌓인 경보 18건**을 읽혀 요약하고, 위험한 것부터 줄 세워, 사람이 읽는 보고서로 만듭니다.

오전에는 보고서의 **형식**을 만들고, 경보를 **묶음으로 나눠 요약**하고, **위험도순으로 정렬**합니다. 오후에는 그 결과로 보고서를 완성합니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저 · 이 파일을 놓는 자리

1. 아침 과제 6 에서 깃허브와 연결한 `security-agent-toolkit` 폴더의 **`agent_core` 폴더**에 이 파일을 넣습니다.
2. VS Code 에서 새 `security-agent-toolkit` 폴더를 열고, 왼쪽 목록의 `agent_core` 에서 이 파일을 엽니다.
3. 오른쪽 위 **커널 선택(Select Kernel)** 을 눌러 **아침 과제 6 에서 새로 만든 가상환경**을 고릅니다. 이름에 `.venv` 처럼 가상환경 이름이 붙은 항목입니다.
4. 셀 실행은 **Shift + Enter** 입니다. 셀은 **위에서 아래로** 차례로 실행합니다.
5. 바로 아래 **위치 확인 셀**을 실행합니다. `준비 완료` 가 나와야 합니다.

### ⚠ 중요 · 무료 키는 1분에 15번까지만 호출할 수 있습니다

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px; border-left:6px solid #e65100">
<b>무료 Gemini API 키의 한도는 1분에 15번입니다.</b><br>
1. 한도를 넘으면 답을 받지 못합니다. 1분 기다린 뒤 같은 셀을 다시 실행합니다.<br>
2. 오늘 오전에 LLM 을 부르는 문제는 <b>3교시의 두 문제</b>뿐입니다. 문제 3-5 는 한 번 실행에 세 번 부릅니다.<br>
3. 한도에 걸리면 <code>[LLM 호출 실패] 상태 코드 429</code> 가 먼저 보이고, 그 아래에 <code>KeyError: 'choices'</code> 나 <code>TypeError: 'NoneType' object is not subscriptable</code> 같은 에러가 이어질 수 있습니다. <b>코드가 틀린 것이 아닙니다.</b> 1분 기다린 뒤 같은 셀을 다시 실행합니다.<br>
4. LLM 의 답 문장과 위험도는 실행할 때마다 달라질 수 있습니다. 🎯 와 글자가 같지 않아도 <b>모양이 같으면</b> 맞습니다.
</div>

### 0.2 오늘 오전의 순서

| 교시 | 무엇 | LLM |
|---|---|---|
| 1교시 | 아침 과제 6 — 저장소 연결 | — |
| 2교시 | 보고서 형식 — 마크다운 · 파일 저장 | 부르지 않음 |
| 3교시 | 묶음으로 나눠 요약하기 | 부름 |
| 4교시 | 위험한 것부터 정렬 · `event_summarizer.py` | 부르지 않음 |

### 0.3 문제를 푸는 법

1. 코드 셀의 번호 주석이 **무엇을 쓸지** 알려 줍니다. 주석 바로 아래 빈 줄에 코드를 씁니다.
2. 주석에 적힌 **변수 이름을 그대로** 씁니다.
3. 미리 채워 둔 줄은 앞에서 배운 것입니다. 고치지 않습니다.
4. 막히면 문제 아래 **💡 힌트**를 보고, 그래도 막히면 노트북 맨 아래 「정답」을 봅니다. 도전 문제는 안 풀고 넘어가도 됩니다.
5. `NameError` 가 나오면 위쪽 셀을 건너뛴 것입니다. 그 이름을 만드는 셀을 먼저 실행합니다.

In [1]:
# 위치 확인 셀 — 그대로 실행합니다
import os                                                     # 폴더 · 파일을 다루는 도구
import sys                                                    # 지금 실행 중인 파이썬의 정보를 주는 도구


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                                        # 다섯 칸 위까지 찾는다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 돌려준다
        folder = os.path.dirname(folder)                      # 한 칸 위 폴더로 올라간다
    return None                                               # 못 읽으면 None


ENV_PATH = find_env()                                         # 찾은 .env 경로. 못 찾으면 None

print("노트북이 실행되는 폴더:", os.getcwd())
print("지금 쓰는 파이썬     :", sys.executable)

if ENV_PATH:
    print("키 파일(.env)         :", ENV_PATH)
    print("준비 완료")
else:                                                         # 아니면
    print(".env 를 찾지 못했습니다")
    print("1) 이 노트북을 새 security-agent-toolkit 폴더 안의 agent_core 폴더로 옮긴 뒤 다시 엽니다")
    print("2) 아침 과제 6 의 「.env 옮기기」를 했는지 봅니다")

노트북이 실행되는 폴더: c:\Users\User\Documents\security-agent-toolkit
지금 쓰는 파이썬     : c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\python.exe
키 파일(.env)         : c:\Users\User\Documents\security-agent-toolkit\.env
준비 완료


아래 두 셀도 실행합니다. 첫 셀은 10/6 오전의 `llm_client.py` 를 같은 내용으로 다시 만듭니다. 10/6 을 못 끝냈어도 오늘 실습을 할 수 있습니다. 둘째 셀은 오늘 쓰는 경보 18건을 파일로 저장합니다.

In [2]:
%%writefile llm_client.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import os                                                     # 폴더 · 파일을 다루는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# Gemini 에 요청을 보낼 주소
URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
MODEL = "gemini-3.5-flash-lite"                               # 쓸 모델 이름


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                          # 지금 폴더에서 다섯 칸 위까지 찾습니다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 돌려준다
        folder = os.path.dirname(folder)        # 한 칸 위 폴더
    return None                                               # 못 읽으면 None


def read_api_key():                                           # .env 에서 키를 읽어 돌려주는 함수
    api_key = None                                            # 키를 못 찾으면 None 으로 남는다
    with open(find_env(), encoding="utf-8") as f:             # .env 를 찾아 읽기로 연다
        for line in f:                                        # 파일을 한 줄씩 꺼낸다
            parts = line.strip().split("=", 1)                # 줄 끝 공백을 지우고 = 에서 둘로 나눈다
            if parts[0] == "GEMINI_API_KEY":                  # = 앞부분이 키 이름이면
                api_key = parts[1]                            # = 뒷부분이 키 값이다
    return api_key                                            # 읽은 키를 돌려준다


def call_llm(question):                                       # 질문을 받아 답 문장을 돌려주는 함수
    # 요청 헤더 — .env 에서 읽은 키를 넣는다
    headers = {"Authorization": "Bearer " + read_api_key(), "Content-Type": "application/json"}
    # 요청 본문 — 모델 · temperature 0 · 질문
    body = {"model": MODEL, "temperature": 0, "messages": [{"role": "user", "content": question}]}
    # POST 요청을 보낸다. 30초 안에 답이 없으면 멈춘다
    response = requests.post(URL, headers=headers, json=body, timeout=30)
    if response.status_code != 200:                           # 200(성공)이 아니면
        print("[LLM 호출 실패] 상태 코드", response.status_code)      # 429 면 1분 기다린 뒤 다시 실행
        return ""                                             # 실패하면 빈 문자열을 돌려준다
    data = response.json()                                    # 응답(JSON 문자열)을 딕셔너리로 바꾼다
    return data["choices"][0]["message"]["content"]           # 답 문장만 돌려준다


def parse_llm_json(text):                                     # LLM 의 답을 딕셔너리로 바꾸는 함수. 못 읽으면 None
    clean = text.replace("```json", "")                       # 코드 블록 표시 ```json 을 지운다
    clean = clean.replace("```", "")                          # 남은 ``` 를 지운다
    clean = clean.strip()                                     # 앞뒤 공백 · 줄바꿈을 지운다

    try:                                                      # 아래 줄을 해 본다
        return json.loads(clean)                              # 문자열을 딕셔너리로 바꿔 돌려준다
    except json.JSONDecodeError:                              # JSON 으로 읽지 못하면 여기로 온다
        return None                                           # 못 읽으면 None

Writing llm_client.py


In [3]:
# 준비 셀 — 그대로 실행합니다. 오늘 쓰는 경보 18건을 events_1007.json 으로 저장합니다
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

events = [                                                    # 오늘 쓰는 경보 18건
    {"id": "E01", "rule": "brute_force", "target": "admin", "time": "03:15", "detail": "211.45.12.9 에서 admin 로그인 실패 4회"},
    {"id": "E02", "rule": "night_login", "target": "admin", "time": "03:17", "detail": "실패 4회 직후 같은 IP 에서 admin 로그인 성공"},
    {"id": "E03", "rule": "password_spraying", "target": "185.220.101.34", "time": "22:15", "detail": "한 IP 가 계정 4개에 차례로 로그인 실패"},
    {"id": "E04", "rule": "brute_force", "target": "root", "time": "02:15", "detail": "198.51.100.23 에서 root 로그인 실패 6회"},
    {"id": "E05", "rule": "permission_denied", "target": "svc_backup", "time": "14:31", "detail": "svc_backup 이 /etc/sudoers 접근 시도, 거부됨"},
    {"id": "E06", "rule": "failed_login", "target": "kim01", "time": "09:12", "detail": "kim01 로그인 실패 1회"},
    {"id": "E07", "rule": "failed_login", "target": "park.js", "time": "10:03", "detail": "park.js 로그인 실패 1회 뒤 바로 성공"},
    {"id": "E08", "rule": "night_login", "target": "deploy", "time": "23:58", "detail": "deploy 계정 토큰 만료 경고"},
    {"id": "E09", "rule": "brute_force", "target": "guest", "time": "23:40", "detail": "203.0.113.61 에서 guest 로그인 실패 5회"},
    {"id": "E10", "rule": "new_ip_login", "target": "lee.yh", "time": "09:02", "detail": "lee.yh 가 처음 보는 IP 192.0.2.17 에서 로그인"},
    {"id": "E11", "rule": "failed_login", "target": "choi.mk", "time": "11:20", "detail": "choi.mk 로그인 실패 2회"},
    {"id": "E12", "rule": "permission_denied", "target": "guest01", "time": "13:07", "detail": "guest01 이 /etc/shadow 접근 시도, 거부됨"},
    {"id": "E13", "rule": "port_scan", "target": "198.51.100.7", "time": "01:30", "detail": "한 IP 가 1분 동안 포트 120개에 접속 시도"},
    {"id": "E14", "rule": "failed_login", "target": "jung.hw", "time": "14:05", "detail": "jung.hw 로그인 실패 1회"},
    {"id": "E15", "rule": "night_login", "target": "svc_batch", "time": "02:41", "detail": "svc_batch 새벽 로그인 실패"},
    {"id": "E16", "rule": "account_created", "target": "tmp_admin", "time": "03:20", "detail": "admin 로그인 직후 관리자 권한 계정 tmp_admin 생성"},
    {"id": "E17", "rule": "failed_login", "target": "kim.cs", "time": "22:14", "detail": "kim.cs 로그인 실패 1회"},
    {"id": "E18", "rule": "config_changed", "target": "firewall", "time": "18:00", "detail": "점검 시간에 ops.kim 이 방화벽 규칙 1개 수정"},
]

with open("events_1007.json", "w", encoding="utf-8") as f:   # 파일을 쓰기로 연다
    json.dump(events, f, ensure_ascii=False, indent=2)        # 한글 그대로, 두 칸 들여쓰기로 저장한다

print(f"경보 {len(events)}건을 events_1007.json 에 저장했습니다")

경보 18건을 events_1007.json 에 저장했습니다


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">2교시 (10:00–10:50) · 보고서 형식 — 마크다운과 파일 저장</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **마크다운 (Markdown)** | `#` · `-` · `**` 를 쓰면 화면에 어떻게 보이나 |
| **보고서 자동화** | 보안관제 일일 보고서에는 보통 무엇이 들어가나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 마크다운 →
- 보고서 자동화 →

### 왜 필요한가

1. 10/6 에 만든 `agent_result.json` 은 **기계가 읽는 기록**입니다. 팀장이 회의 전에 한 장으로 읽기는 어렵습니다.
2. 사람에게 보여 줄 것은 **제목 · 소제목 · 목록**이 있는 문서입니다. 그 형식이 마크다운입니다.
3. 오늘 오후에 완성할 보고서도 마크다운 파일(`.md`)입니다. 2교시에는 그 **형식과 저장**만 먼저 익힙니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.1 마크다운은 기호 몇 개로 문서의 모양을 정합니다</mark>

| 쓰는 것 | 보이는 모양 | 보고서에서 |
|---|---|---|
| `# 제목` | 가장 큰 제목 | 보고서 이름 |
| `## 소제목` | 그다음 크기의 제목 | 「한눈에 보기」·「건별 내역」 |
| `- 내용` | 점이 붙은 목록 | 경보 한 건 한 줄 |
| `**굵게**` | 굵은 글씨 | 주의 문구 |

```python
title = "# 야간 보안 관제 보고 (2026-10-07)"
lines = "- E01 admin 로그인 실패 4회\n"          # \n 은 줄바꿈
lines = lines + "- E02 admin 로그인 성공\n"

report = title + "\n\n## 건별 내역\n" + lines
```

1. 보고서도 결국 **문자열 하나**입니다. 줄마다 끝에 `\n`(줄바꿈)을 붙여 이어 붙입니다.
2. 저장은 9/23 오후에 배운 `open(파일, "w", encoding="utf-8")` 과 `f.write()` 입니다.
3. VS Code 에서 `.md` 파일을 열고 `Ctrl+Shift+V` 를 누르면 꾸민 모양(미리 보기)으로 보입니다.

#### 🐍 문법 상자 · `\n` 은 줄바꿈 글자입니다

```python
line = "- E01 admin 로그인 실패\n"
print(len("\n"))          # 1 — 두 글자로 보이지만 글자 하나
```

| 쓰는 것 | 화면 | 뜻 |
|---|---|---|
| `"a\nb"` | `a` 와 `b` 가 두 줄로 | `\n` 자리에서 줄이 바뀐다 |
| `"a" + "b"` | `ab` | 줄바꿈 없이 붙는다 |

⚠ 목록 줄 끝에 `\n` 을 빠뜨리면 다음 경보가 같은 줄에 붙습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
items = ["brute_force", "night_login"]

text = "## 오늘의 경보\n"
for item in items:
    text = text + "- " + item + "\n"

print(text)
```

막히면 바로 위 `2.1 마크다운은 기호 몇 개로 문서의 모양을 정합니다` 설명을 다시 봅니다.

In [4]:
items = ["brute_force", "night_login"]

text = "## 오늘의 경보\n"
for item in items:
    text = text + "- " + item + "\n"

print(text)

## 오늘의 경보
- brute_force
- night_login



✅ `## 오늘의 경보` · `- brute_force` · `- night_login` 세 줄 — `\n` 마다 줄이 바뀝니다

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-2 · 보고서 제목 만들기</font></h3></td></tr></table>

날짜를 넣어 보고서의 제목 줄을 만드시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `# 야간 보안 관제 보고 (2026-10-07)` |

**💡 힌트**

1. f-string 은 `f"…{변수}…"` 꼴입니다.
2. 맨 앞의 `#` 와 빈칸 한 칸이 마크다운 제목 표시입니다.
3. 괄호 `( )` 는 글자 그대로 넣습니다.

In [6]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
today = "2026-10-07"                                          # 보고서 날짜
# ── 여기까지 ──

# 1. title 이라는 변수에 f-string 으로 "# 야간 보안 관제 보고 (" + today + ")" 꼴의 제목을 담으세요
title = f"# 야간 보안 관제 보고 ({today})"

print(title)                                                  # 만든 제목을 본다

# 야간 보안 관제 보고 (2026-10-07)


#### 🐍 문법 상자 · f-string 안에서 딕셔너리 값 꺼내기

```python
event = {"id": "E01", "rule": "brute_force"}
print(f"- [{event['rule']}] {event['id']}")     # - [brute_force] E01
```

| 쓰는 것 | 결과 | 이유 |
|---|---|---|
| `f"{event['id']}"` | 된다 | 바깥이 큰따옴표라 안쪽은 작은따옴표 |
| `f"{event["id"]}"` | 파이썬 3.11 이하에서는 문법 오류 | 큰따옴표가 바깥 문자열을 닫아 버린다. 3.12 부터는 되지만 읽기 헷갈린다 |

⚠ 바깥과 안쪽의 따옴표 종류를 **다르게** 씁니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-3 · 경보를 목록 줄로 바꾸기</font></h3></td></tr></table>

경보 18건을 `- [룰] id 내용` 꼴의 목록 줄로 이어 붙이시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `- [brute_force] E01 211.45.12.9 에서 admin 로그인 실패 4회` 로 시작하는 18줄 |

**💡 힌트**

1. 빈 문자열 `lines` 뒤에 한 줄씩 이어 붙입니다.
2. 줄 끝에 `\n` 을 붙여야 다음 경보가 새 줄에 나옵니다.
3. f-string 안에서 딕셔너리 값을 꺼낼 때는 작은따옴표 `event['id']` 를 씁니다.

In [7]:
import json

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("events_1007.json", encoding="utf-8") as f:
    events = json.load(f)
lines = ""
# — 여기까지 —

# 1. for 로 events 에서 경보를 하나씩 꺼내 event 에 담으세요
for event in events:
    # 2. lines 뒤에 f"- [{event['rule']}] {event['id']} {event['detail']}\n" 을 이어 붙이세요
    lines += f"- [{event['rule']}] {event['id']} {event['detail']}\n"

print(lines)

- [brute_force] E01 211.45.12.9 에서 admin 로그인 실패 4회
- [night_login] E02 실패 4회 직후 같은 IP 에서 admin 로그인 성공
- [password_spraying] E03 한 IP 가 계정 4개에 차례로 로그인 실패
- [brute_force] E04 198.51.100.23 에서 root 로그인 실패 6회
- [permission_denied] E05 svc_backup 이 /etc/sudoers 접근 시도, 거부됨
- [failed_login] E06 kim01 로그인 실패 1회
- [failed_login] E07 park.js 로그인 실패 1회 뒤 바로 성공
- [night_login] E08 deploy 계정 토큰 만료 경고
- [brute_force] E09 203.0.113.61 에서 guest 로그인 실패 5회
- [new_ip_login] E10 lee.yh 가 처음 보는 IP 192.0.2.17 에서 로그인
- [failed_login] E11 choi.mk 로그인 실패 2회
- [permission_denied] E12 guest01 이 /etc/shadow 접근 시도, 거부됨
- [port_scan] E13 한 IP 가 1분 동안 포트 120개에 접속 시도
- [failed_login] E14 jung.hw 로그인 실패 1회
- [night_login] E15 svc_batch 새벽 로그인 실패
- [account_created] E16 admin 로그인 직후 관리자 권한 계정 tmp_admin 생성
- [failed_login] E17 kim.cs 로그인 실패 1회
- [config_changed] E18 점검 시간에 ops.kim 이 방화벽 규칙 1개 수정



#### 🐍 문법 상자 · 파일 열기 모드 `"w"` 와 `"a"`

```python
with open("report_test.md", "w", encoding="utf-8") as f:   # 덮어쓰기
    f.write(report)
```

| 모드 | 파일이 이미 있으면 | 쓰는 때 |
|---|---|---|
| `"w"` | 내용을 **지우고** 새로 쓴다 | 보고서처럼 매번 새로 만드는 파일 |
| `"a"` | 기존 내용 **뒤에 이어 쓴다** | 로그처럼 계속 쌓는 파일 |
| `"r"` (적지 않으면 이것) | 읽기만 한다 | 파일을 읽을 때 |

노트북의 `%%writefile` 도 `"w"` 처럼 덮어씁니다. 이어 쓰려면 `%%writefile -a` 입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-4 · 보고서를 파일로 저장하기</font></h3></td></tr></table>

제목 · 소제목 · 목록을 이어 붙여 `report_test.md` 로 저장하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `report_test.md 를 저장했습니다` — 왼쪽 목록에 파일이 생기고, 열어서 `Ctrl+Shift+V` 로 보면 제목과 목록이 보입니다 |

**💡 힌트**

1. 이어 붙이는 순서는 제목 → 소제목 → 목록입니다.
2. 쓰기로 열 때는 `"w"` 와 `encoding="utf-8"` 을 넣습니다.
3. 파일에 쓰는 것은 `f.write(report)` 입니다.

In [8]:
import json

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("events_1007.json", encoding="utf-8") as f:
    events = json.load(f)
title = "# 야간 보안 관제 보고 (2026-10-07)"
lines = ""
for event in events:
    # 「- [룰] id 내용」 한 줄을 붙인다
    lines = lines + f"- [{event['rule']}] {event['id']} {event['detail']}\n"
# — 여기까지 —

# 1. report 라는 변수에 title, "\n\n## 건별 내역\n", lines 를 이어 붙여 담으세요
report = title + "\n\n## 건별 내역\n" + lines

# 2. "report_test.md" 를 쓰기로 열고(encoding="utf-8") report 를 저장하세요 (with 문, f.write)
with open("report_test.md", "w", encoding="utf-8") as f:
    f.write(report)

print("report_test.md 를 저장했습니다. 왼쪽 목록에서 열어 봅니다")

report_test.md 를 저장했습니다. 왼쪽 목록에서 열어 봅니다


#### 🐍 문법 상자 · `.get(키, 0)` — 없는 키도 셀 수 있게

```python
count = {"brute_force": 2}
print(count.get("brute_force", 0))      # 2
print(count.get("port_scan", 0))        # 0 — 없는 키라 0
```

| 쓰는 것 | 키가 있을 때 | 키가 없을 때 |
|---|---|---|
| `count[rule]` | 값 | **`KeyError` 로 멈춘다** |
| `count.get(rule, 0)` | 값 | `0` |

그래서 `count[rule] = count.get(rule, 0) + 1` 한 줄로 처음 보는 룰도 1 부터 셉니다. 9/23 에 배웠습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 2-5 · 룰마다 몇 건인지 세기</font></h3></td></tr></table>

경보를 룰 이름별로 세어 `- 룰: N건` 꼴로 출력하시오. 보고서의 「한눈에 보기」에 들어갈 숫자입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `- brute_force: 3건` · `- night_login: 3건` · … · `- failed_login: 5건` 등 아홉 줄 |

**💡 힌트**

1. 9/23 에 배운 딕셔너리로 셉니다. 키는 룰 이름, 값은 건수입니다.
2. `count.get(rule, 0)` 은 처음 보는 룰이면 0 을 돌려줍니다.
3. **숫자는 코드가 셉니다.** LLM 에게 세게 하지 않습니다 — 틀릴 수 있습니다.

In [9]:
import json

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("events_1007.json", encoding="utf-8") as f:
    events = json.load(f)
count = {}
# — 여기까지 —

# 1. for 로 events 에서 경보를 하나씩 꺼내 event 에 담으세요
for event in events:
    # 2. rule 이라는 변수에 event 의 rule 값을 담으세요
    rule = event["rule"]
    # 3. count 의 rule 키 값을 1 늘리세요. 처음 보는 룰이면 1 로 시작합니다 (.get(rule, 0) + 1)
    count[rule] = count.get(rule, 0) + 1

for rule in count:
    print(f"- {rule}: {count[rule]}건")

- brute_force: 3건
- night_login: 3건
- password_spraying: 1건
- permission_denied: 2건
- failed_login: 5건
- new_ip_login: 1건
- port_scan: 1건
- account_created: 1건
- config_changed: 1건


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 2-6 · 룰별로 묶은 보고서 파일 만들기</font></h3></td></tr></table>

룰 이름마다 소제목을 두고 그 아래에 경보를 모은 보고서를, 날짜가 든 파일 이름으로 저장하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `daily_report_20261007.md` — 파일을 열면 `## brute_force` 아래 세 건, `## night_login` 아래 세 건 … |

**💡 힌트**

1. 룰 이름을 한 번씩만 모은 리스트를 먼저 만듭니다. `not in` 으로 이미 넣었는지 확인합니다.
2. 바깥 반복은 룰, 안쪽 반복은 경보입니다.
3. `today.replace("-", "")` 는 `20261007` 입니다.

In [10]:
import json

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("events_1007.json", encoding="utf-8") as f:
    events = json.load(f)
count = {}
# — 여기까지 —

# 1. for 로 events 에서 경보를 하나씩 꺼내 event 에 담으세요
for event in events:
    # 2. rule 이라는 변수에 event 의 rule 값을 담으세요 (여기 들여쓰기 필요)
    rule = event["rule"]
    
    # 3. count 의 rule 키 값을 1 늘리세요. (여기 들여쓰기 필요)
    count[rule] = count.get(rule, 0) + 1

for rule in count:
    print(f"- {rule}: {count[rule]}건")

- brute_force: 3건
- night_login: 3건
- password_spraying: 1건
- permission_denied: 2건
- failed_login: 5건
- new_ip_login: 1건
- port_scan: 1건
- account_created: 1건
- config_changed: 1건


### 📋 2교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 제목 · 소제목 · 목록 | `# ` · `## ` · `- ` |
| 줄을 이어 붙인다 | `lines = lines + f"- …\n"` |
| 파일로 저장한다 | `with open("report.md", "w", encoding="utf-8") as f:` → `f.write(report)` |
| 건수를 센다 | `count[rule] = count.get(rule, 0) + 1` — 숫자는 코드가 센다 |

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">3교시 (11:00–11:50) · 묶음으로 나눠 요약하기</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **배치 처리 (batch)** | 많은 데이터를 한 번에 처리하지 않고 나눠 처리하는 이유 |
| **컨텍스트 길이 (context length)** | LLM 에 한 번에 넣을 수 있는 글의 양에 한도가 있는 이유 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 배치 처리 →
- 컨텍스트 길이 →

### 왜 필요한가

1. 실제 관제에서는 밤사이 경보가 수백 건 쌓입니다. 한 번에 다 넣으면 LLM 이 받을 수 있는 길이를 넘거나, 중간 경보를 빠뜨립니다.
2. 그래서 **몇 건씩 묶음으로 나눠** 요약을 받고, 받은 요약을 하나로 모읍니다.
3. 오늘은 18건을 **6건씩 세 묶음**으로 나눕니다. 한 번 실행에 LLM 을 세 번 부르므로 무료 한도 안입니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.1 리스트를 잘라 묶음을 만듭니다</mark>

```python
events[0:6]      # 0번부터 6번 앞까지 — 여섯 건
events[6:12]     # 6번부터 12번 앞까지

for start in range(0, 18, 6):        # start = 0, 6, 12
    batch = events[start:start + 6]
```

| 쓰는 것 | 뜻 |
|---|---|
| `리스트[a:b]` | a 번부터 b 번 **앞까지** 잘라 새 리스트를 만든다 (9/22 에 배운 자르기) |
| `range(0, 18, 6)` | 0 에서 시작해 6씩 건너뛰며 18 앞까지 — `0, 6, 12` |
| `start + 6` | 이번 묶음이 끝나는 자리 |

마지막 묶음이 여섯 건이 안 되어도 괜찮습니다. 자르기는 있는 만큼만 잘라 줍니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
nums = [10, 20, 30, 40, 50, 60, 70]

for start in range(0, len(nums), 3):
    print(nums[start:start + 3])
```

막히면 바로 위 `3.1 리스트를 잘라 묶음을 만듭니다` 설명을 다시 봅니다.

In [ ]:
nums = [10, 20, 30, 40, 50, 60, 70]

for start in range(0, len(nums), 3):
    print(nums[start:start + 3])

✅ `[10, 20, 30]` · `[40, 50, 60]` · `[70]` — 마지막 묶음은 남은 만큼만

#### 🐍 문법 상자 · 리스트 자르기 `[a:b]`

```python
ids = ["E01", "E02", "E03", "E04", "E05"]
print(ids[0:2])      # ['E01', 'E02'] — 2번은 들어가지 않는다
print(ids[-1])       # E05 — 마지막
print(ids[3:10])     # ['E04', 'E05'] — 넘치게 잘라도 오류가 없다
```

| 쓰는 것 | 결과 | 뜻 |
|---|---|---|
| `ids[a:b]` | a 번부터 **b 번 앞까지** | 끝 번호는 포함하지 않는다 |
| `ids[-1]` | 마지막 하나 | 뒤에서 첫째 |
| `ids[3:10]` | 있는 만큼만 | 마지막 묶음이 모자라도 괜찮다 |

⚠ `[0:6]` 은 여섯 개(0~5번)입니다. 일곱 개가 아닙니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-2 · 첫 묶음 꺼내기</font></h3></td></tr></table>

경보 18건 가운데 첫 여섯 건을 잘라 묶음 `batch` 를 만드시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `6 E01 E06` |

**💡 힌트**

1. 자르기는 `events[0:6]` 입니다.
2. 마지막 원소는 `batch[-1]` 로 꺼냅니다.
3. 출력은 길이 · 첫 id · 마지막 id 를 한 줄에 씁니다.

In [11]:
import json

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("events_1007.json", encoding="utf-8") as f:
    events = json.load(f)
today = "2026-10-07"
# — 여기까지 —

# 1. filename 에 f"daily_report_{today}.md" 를 담으세요. 날짜의 - 는 빼고 20261007 꼴로 만듭니다 (.replace)
date_str = today.replace("-", "")
filename = f"daily_report_{date_str}.md"

# 2. 룰 이름마다 "## 룰이름" 소제목을 한 번만 쓰고, 그 아래에 그 룰의 경보를 "- id detail" 로 모으세요
# (힌트 1) 룰 이름을 중복 없이 모은 리스트 생성
rule_list = []
for event in events:
    if event["rule"] not in rule_list:
        rule_list.append(event["rule"])

# (힌트 2) 바깥 반복은 룰, 안쪽 반복은 경보
report = ""
for rule in rule_list:
    report += f"## {rule}\n"
    for event in events:
        if event["rule"] == rule:
            report += f"- {event['id']} {event['detail']}\n"
    report += "\n"  # 룰 그룹 간 구분을 위한 줄바꿈

# 3. 만든 보고서를 filename 으로 저장하고, filename 을 출력하세요
with open(filename, "w", encoding="utf-8") as f:
    f.write(report)

print(filename)

daily_report_20261007.md


#### 🐍 문법 상자 · `range` 의 인자 하나 · 둘 · 셋

```python
print(list(range(4)))          # [0, 1, 2, 3]
print(list(range(2, 5)))       # [2, 3, 4]
print(list(range(0, 18, 6)))   # [0, 6, 12]
```

| 쓰는 것 | 나오는 수 | 뜻 |
|---|---|---|
| `range(끝)` | 0 부터 끝 앞까지 | 지금까지 쓴 모양 |
| `range(시작, 끝)` | 시작부터 끝 앞까지 | |
| `range(시작, 끝, 간격)` | 시작부터 간격씩 건너뛰며 | **오늘 새로 쓰는 모양** |

끝 번호는 자르기처럼 **포함하지 않습니다.** `range(0, 18, 6)` 에 18 은 나오지 않습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-3 · 묶음 세 개로 나누기</font></h3></td></tr></table>

경보를 여섯 건씩 나눠, 묶음마다 시작 번호 · 길이 · 첫 id 를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `0 6 E01` · `6 6 E07` · `12 6 E13` |

**💡 힌트**

1. `range(0, len(events), size)` 가 0, 6, 12 를 차례로 줍니다.
2. 끝나는 자리는 `start + size` 입니다.
3. 아직 LLM 은 부르지 않습니다. 나누는 것만 확인합니다.

In [12]:
import json

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("events_1007.json", encoding="utf-8") as f:
    events = json.load(f)
# — 여기까지 —

# 1. batch 라는 변수에 events 의 0번부터 6개(0:6)를 잘라 담으세요
batch = events[0:6]

# 2. batch 의 길이와, 첫 경보와 마지막 경보의 id 를 출력하세요
print(len(batch), batch[0]['id'], batch[-1]['id'])

6 E01 E06


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.2 묶음을 JSON 문자열로 바꿔 LLM 에 보냅니다</mark>

```python
instruction = "다음 보안 경보 목록을 읽고, 경보마다 id, risk_level, summary 세 키를 가진 JSON 배열로만 답하세요. …"
batch_text = json.dumps(batch, ensure_ascii=False)     # 리스트 → JSON 문자열

text = call_llm(instruction + batch_text)
result = parse_llm_json(text)                          # JSON 배열 → 리스트
```

1. `json.dumps` 는 9/28 에 배운 `json.dump` 의 짝입니다. 파일 대신 **문자열**로 바꿔 줍니다.
2. 답을 **JSON 배열**로 달라고 했으므로, `parse_llm_json` 이 돌려주는 것은 딕셔너리가 아니라 **리스트**입니다.
3. 경보마다 `risk_level`(위험도)을 LLM 이 정합니다. 숫자 세기와 달리 판단은 LLM 에 맡깁니다. 다만 틀릴 수 있어서 4교시와 오후에 사람이 보기 좋게 정리합니다.

#### 🐍 문법 상자 · `json` 의 네 함수 구별하기

| 함수 | 무엇을 → 무엇으로 | 오늘 쓰는 곳 |
|---|---|---|
| `json.dump(값, f)` | 파이썬 값 → **파일** | 요약을 `event_summaries.json` 에 저장 |
| `json.load(f)` | **파일** → 파이썬 값 | `events_1007.json` 읽기 |
| `json.dumps(값)` | 파이썬 값 → **문자열** | 묶음을 LLM 에 보낼 글로 바꾸기 |
| `json.loads(문자열)` | **문자열** → 파이썬 값 | `parse_llm_json` 안에서 |

- 끝의 **`s` 는 string(문자열)** 입니다. `s` 가 있으면 문자열, 없으면 파일입니다.
- `ensure_ascii=False` 는 한글을 `\uad00` 같은 기호로 바꾸지 않고 그대로 둡니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-4 · 첫 묶음을 LLM 으로 요약하기</font></h3></td></tr></table>

첫 묶음(여섯 건)을 LLM 에 보내 요약 리스트를 받고, 길이와 첫 요약을 출력하시오. **LLM 을 한 번 부릅니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `6` 그리고 `{'id': 'E01', 'risk_level': '…', 'summary': '…'}` 꼴의 딕셔너리 하나. 위험도와 문장은 실행할 때마다 다를 수 있습니다 |

**💡 힌트**

1. 질문은 `instruction + batch_text` 입니다.
2. 받은 문자열을 `parse_llm_json` 으로 바꿉니다.
3. `TypeError: object of type 'NoneType'` 이 나오면 답을 읽지 못한 것입니다. 1분 뒤 다시 실행합니다.

In [13]:
import json

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("events_1007.json", encoding="utf-8") as f:
    events = json.load(f)
size = 6
# — 여기까지 —

# 1. for 로 range(0, len(events), size) 를 돌면서 start 에 담으세요
for start in range(0, len(events), size):
    # 2. batch 에 events 의 start 부터 start + size 앞까지 잘라 담으세요
    batch = events[start : start + size]
    
    # 3. start 와 batch 의 길이와 batch 첫 경보의 id 를 출력하세요
    print(start, len(batch), batch[0]['id'])

0 6 E01
6 6 E07
12 6 E13


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.3 묶음 하나를 요약하는 줄을 함수 `summarize_batch` 로 묶습니다</mark>

문제 3-4 의 「JSON 문자열로 바꾸고 → 보내고 → 리스트로 바꾸는」 줄을 묶음마다 되풀이합니다. 함수 하나로 묶어 둡니다.

| 함수 안의 줄 | 하는 일 | 문제 3-4 에서 |
|---|---|---|
| `batch_text = json.dumps(batch, ensure_ascii=False)` | 묶음을 문자열로 바꾼다 | 미리 채운 줄 |
| `text = call_llm(instruction + batch_text)` | LLM 에 보낸다 | 1번 |
| `if text == "":` … `return []` | 답을 못 받으면 빈 리스트 | — |
| `result = parse_llm_json(text)` | 리스트로 바꾼다 | 2번 |
| `if result is None:` … `return []` | 못 읽으면 빈 리스트 | — |

- 실패해도 `None` 이 아니라 **빈 리스트**를 돌려줍니다. 그래서 묶음 하나가 실패해도 다음 묶음은 계속 요약됩니다.
- 쓰는 법은 `result = summarize_batch(batch)` 한 줄입니다.

아래 셀을 실행합니다. 함수를 **만들어 두기만** 합니다.

In [14]:
import json
import importlib
import llm_client

importlib.reload(llm_client)
from llm_client import call_llm, parse_llm_json

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("events_1007.json", encoding="utf-8") as f:
    events = json.load(f)
# 요약 지시 – 출력 형식(JSON 배열)과 키 이름을 정한다
instruction = "다음 보안 경보 목록을 읽고, 경보마다 id, risk_level, summary 세 키를 가진 JSON 배열로만 답하세요. "
# 위험도 세 단어와 요약 길이를 정한다. 뒤에 경보 목록이 붙는다
instruction = instruction + "risk_level 은 high, medium, low 중 하나입니다. summary 는 한국어 한 문장으로 짧게 씁니다. 경보 목록: "
batch = events[0:6]
batch_text = json.dumps(batch, ensure_ascii=False)
# — 여기까지 —

# 1. text 라는 변수에 call_llm(instruction + batch_text) 의 답을 담으세요
text = call_llm(instruction + batch_text)

# 2. result 라는 변수에 parse_llm_json(text) 의 결과를 담으세요
result = parse_llm_json(text)

# 3. result 의 길이와 첫 요약을 출력하세요
print(len(result), result[0])

TypeError: can only concatenate str (not "NoneType") to str

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 3-5 · 세 묶음을 모두 요약해 모으기</font></h3></td></tr></table>

경보를 여섯 건씩 나눠 묶음마다 `summarize_batch` 로 요약하고, 요약을 `summaries` 하나에 모아 파일로 저장하시오. **한 번 실행에 LLM 을 세 번 부릅니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `요약 18건을 event_summaries.json 에 저장했습니다` — 묶음이 하나 실패하면 12건이 나옵니다. 1분 뒤 다시 실행합니다 |

**💡 힌트**

1. 나누는 반복은 문제 3-3 과 같습니다.
2. 받은 요약 리스트를 하나씩 꺼내 `summaries` 에 `append` 합니다.
3. 같은 셀을 연달아 실행하지 않습니다. 한도(1분 15번)를 넘을 수 있습니다.

In [16]:
import json
import importlib
import llm_client

importlib.reload(llm_client)
# summarize_batch 함수를 llm_client에서 가져옵니다
from llm_client import summarize_batch

# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
with open("events_1007.json", encoding="utf-8") as f:
    events = json.load(f)
size = 6
summaries = []
# — 여기까지 —

# 1. for 로 range(0, len(events), size) 를 돌면서 start 에 담으세요
for start in range(0, len(events), size):
    # 2. batch 에 events[start:start + size] 를 담고, result 에 summarize_batch(batch) 의 결과를 담으세요
    batch = events[start : start + size]
    result = summarize_batch(batch)
    
    # 3. for 로 result 에서 하나씩 꺼내 summaries 에 append 하세요
    for item in result:
        summaries.append(item)

with open("event_summaries.json", "w", encoding="utf-8") as f:
    json.dump(summaries, f, ensure_ascii=False, indent=2)

print(f"요약 {len(summaries)}건을 event_summaries.json 에 저장했습니다")

ImportError: cannot import name 'summarize_batch' from 'llm_client' (c:\Users\User\Documents\security-agent-toolkit\llm_client.py)

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 3-6 · LLM 이 빠뜨리거나 지어낸 것 찾기</font></h3></td></tr></table>

저장한 요약을 원래 경보와 맞춰 보고, **요약에만 있는 id**(지어낸 것)와 **요약이 빠진 경보**를 찾으시오. LLM 을 부르지 않습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 대부분 `확인 끝` 한 줄. 빠지거나 지어낸 것이 있으면 그 줄이 먼저 나옵니다 |

**💡 힌트**

1. id 만 모은 리스트를 양쪽에서 하나씩 만듭니다.
2. `not in` 으로 한쪽에 없는 것을 찾습니다.
3. 10/6 2교시의 「LLM 은 지어낼 수 있다」를 코드로 확인하는 문제입니다.

In [17]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1007.json", encoding="utf-8") as f:         # 준비 셀이 저장한 경보 파일을 연다
    events = json.load(f)                                     # 경보 목록을 리스트로 읽는다
with open("event_summaries.json", encoding="utf-8") as f:    # 문제 3-5 가 저장한 요약
    summaries = json.load(f)                                  # 저장한 요약을 읽는다
# ── 여기까지 ──

# 1. events 의 id 를 모두 담은 리스트 known_ids 를 만드세요
# 2. summaries 를 돌면서 id 가 known_ids 에 없는 요약을 "[지어낸 id]" 와 함께 출력하세요
# 3. 반대로, events 가운데 요약이 빠진 경보의 id 를 "[빠진 경보]" 와 함께 출력하세요
# 4. 마지막에 "확인 끝" 을 출력하세요


FileNotFoundError: [Errno 2] No such file or directory: 'event_summaries.json'

### 📋 3교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 리스트를 자른다 | `events[start:start + 6]` |
| 6건씩 건너뛰며 돈다 | `for start in range(0, len(events), 6):` |
| 묶음을 문자열로 바꾼다 | `json.dumps(batch, ensure_ascii=False)` |
| 묶음 하나를 요약한다 | `result = summarize_batch(batch)` — 실패하면 빈 리스트 |
| 요약을 모은다 | `summaries.append(item)` |

판단(위험도 · 요약 문장)은 LLM 이, 나누기와 모으기는 코드가 합니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">4교시 (12:00–12:50) · 위험한 것부터 정렬 · event_summarizer.py</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **트리아지 (triage)** | 보안관제에서 경보를 처리하는 순서를 어떻게 정하나 |
| **대소문자 구분** | `"High"` 와 `"high"` 를 컴퓨터는 같은 글자로 보나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 트리아지 →
- 대소문자 구분 →

### 왜 필요한가

1. 보고서를 받은 사람은 **위험한 것부터** 봐야 합니다. 18건이 시간 순서로 섞여 있으면 high 를 찾느라 시간을 씁니다.
2. LLM 은 위험도를 `high` · `High` · `HIGH` 처럼 **대소문자를 섞어** 쓰기도 합니다. 그대로 비교하면 `"High" == "high"` 가 `False` 라서 빠집니다.
3. 그래서 **소문자로 맞춘 뒤** high → medium → low 순서로 모읍니다. 4교시부터는 3교시의 LLM 결과 대신 **미리 준비한 요약 18건**을 써서, 모두 같은 결과를 봅니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.1 위험도별로 차례로 모으면 정렬이 됩니다</mark>

```python
sorted_summaries = []
for level in ["high", "medium", "low"]:          # 이 순서대로
    for s in summaries:                           # 전체를 한 번씩 훑으며
        if s["risk_level"].lower() == level:      # 소문자로 맞춰 비교
            sorted_summaries.append(s)
```

| 쓰는 것 | 뜻 |
|---|---|
| `["high", "medium", "low"]` | 보고 싶은 순서를 리스트로 적는다 |
| `.lower()` | 글자를 모두 소문자로 바꾼다 — `"High"` → `"high"` |
| 반복 안의 반복 | high 를 전부 모으고, 그다음 medium 을 전부, 그다음 low 를 전부 |

10/2 오후에 배운 「반복 안에 반복」 그대로입니다. 새 문법은 `.lower()` 하나입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
items = [{"risk": "low"}, {"risk": "HIGH"}, {"risk": "medium"}]

for level in ["high", "medium", "low"]:
    for item in items:
        if item["risk"].lower() == level:
            print(item["risk"])
```

막히면 바로 위 `4.1 위험도별로 차례로 모으면 정렬이 됩니다` 설명을 다시 봅니다.

In [ ]:
items = [{"risk": "low"}, {"risk": "HIGH"}, {"risk": "medium"}]

for level in ["high", "medium", "low"]:
    for item in items:
        if item["risk"].lower() == level:
            print(item["risk"])

✅ `HIGH` · `medium` · `low` — 비교는 소문자로 하지만 출력은 원래 글자 그대로입니다

#### 🐍 문법 상자 · 문자열 메서드는 원본을 바꾸지 않습니다

```python
level = "High"
level.lower()             # 새 문자열 "high" 를 만들 뿐
print(level)              # High — 그대로다
level = level.lower()     # 담아야 바뀐다
```

| 메서드 | 결과 | 오늘 쓰는 곳 |
|---|---|---|
| `.lower()` | 모두 소문자 | 위험도 비교 |
| `.upper()` | 모두 대문자 | 보고서의 `[HIGH]` |
| `.strip()` | 앞뒤 공백 · 줄바꿈 제거 | `parse_llm_json` |
| `.replace(a, b)` | a 를 b 로 바꾼 새 문자열 | 날짜의 `-` 빼기 |

비교할 때는 `s["risk_level"].lower() == "high"` 처럼 **그 자리에서** 바꿔 씁니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-2 · high 건수 세기</font></h3></td></tr></table>

요약 18건 가운데 위험도가 high 인 것을 세시오. 대소문자가 섞여 있습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `high 6건` |

**💡 힌트**

1. `.lower()` 로 바꾼 뒤 `"high"` 와 비교합니다.
2. `.lower()` 없이 비교하면 몇 건이 나오는지도 해 봅니다(4건).
3. 이 숫자가 오후 보고서의 「high N건」이 됩니다. 코드가 셉니다.

In [18]:
# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
summaries = [
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 - 탈취 의심"},
    {"id": "E03", "risk_level": "high", "summary": "한 IP 의 비밀번호 스프레이"},
    {"id": "E04", "risk_level": "high", "summary": "root 계정에 로그인 실패 6회"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
    {"id": "E07", "risk_level": "low", "summary": "park.js 실패 뒤 정상 로그인"},
    {"id": "E08", "risk_level": "medium", "summary": "deploy 토큰 만료"},
    {"id": "E09", "risk_level": "medium", "summary": "guest 계정에 로그인 실패 5회"},
    {"id": "E10", "risk_level": "medium", "summary": "lee.yh 새 IP 로그인"},
    {"id": "E11", "risk_level": "low", "summary": "choi.mk 로그인 실패 2회"},
    {"id": "E12", "risk_level": "high", "summary": "guest01 의 shadow 접근 거부"},
    {"id": "E13", "risk_level": "medium", "summary": "외부 IP 의 포트 스캔"},
    {"id": "E14", "risk_level": "low", "summary": "jung.hw 로그인 실패 1회"},
    {"id": "E15", "risk_level": "medium", "summary": "svc_batch 새벽 로그인 실패"},
    {"id": "E16", "risk_level": "HIGH", "summary": "관리자 권한 계정 무단 생성 의심"},
    {"id": "E17", "risk_level": "low", "summary": "kim.cs 로그인 실패 1회"},
    {"id": "E18", "risk_level": "low", "summary": "점검 중 방화벽 규칙 수정"},
]
# — 여기까지 —

high_count = 0

for item in summaries:
    # risk_level 값을 소문자로 변환하여 "high"와 비교
    if item["risk_level"].lower() == "high":
        high_count += 1

print(f"high {high_count}건")

high 6건


#### 🐍 문법 상자 · 반복 안의 반복은 몇 번 도나

```python
for level in ["high", "medium", "low"]:      # 3번
    for s in summaries:                       # 그때마다 18번
        if s["risk_level"].lower() == level:
            sorted_summaries.append(s)
```

| 바깥 반복 | 안쪽에서 하는 일 | 모이는 것 |
|---|---|---|
| `level = "high"` | 18건을 다 훑는다 | high 6건 |
| `level = "medium"` | 18건을 다시 다 훑는다 | medium 6건 |
| `level = "low"` | 18건을 또 다 훑는다 | low 6건 |

모두 3 × 18 = **54번** 비교합니다. 바깥 순서(high → medium → low)가 곧 결과의 순서가 됩니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-3 · 위험도순으로 정렬하기</font></h3></td></tr></table>

요약 18건을 high → medium → low 순서로 모아 `sorted_summaries` 를 만드시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `E01 high` · `E02 High` · `E03 high` · `E04 high` · `E12 high` · `E16 HIGH` · `E05 medium` |

**💡 힌트**

1. 바깥 반복은 위험도, 안쪽 반복은 요약입니다.
2. `.lower()` 로 맞춰 비교합니다.
3. 같은 위험도 안에서는 원래 순서(id 순)가 유지됩니다.

In [19]:
# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
summaries = [
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 - 탈취 의심"},
    {"id": "E03", "risk_level": "high", "summary": "한 IP 의 비밀번호 스프레이"},
    {"id": "E04", "risk_level": "high", "summary": "root 계정에 로그인 실패 6회"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
    {"id": "E07", "risk_level": "low", "summary": "park.js 실패 뒤 정상 로그인"},
    {"id": "E08", "risk_level": "medium", "summary": "deploy 토큰 만료"},
    {"id": "E09", "risk_level": "medium", "summary": "guest 계정에 로그인 실패 5회"},
    {"id": "E10", "risk_level": "medium", "summary": "lee.yh 새 IP 로그인"},
    {"id": "E11", "risk_level": "low", "summary": "choi.mk 로그인 실패 2회"},
    {"id": "E12", "risk_level": "high", "summary": "guest01 의 shadow 접근 거부"},
    {"id": "E13", "risk_level": "Medium", "summary": "외부 IP 의 포트 스캔"},
    {"id": "E14", "risk_level": "low", "summary": "jung.hw 로그인 실패 1회"},
    {"id": "E15", "risk_level": "medium", "summary": "svc_batch 새벽 로그인 실패"},
    {"id": "E16", "risk_level": "HIGH", "summary": "관리자 권한 계정 무단 생성 의심"},
    {"id": "E17", "risk_level": "low", "summary": "kim.cs 로그인 실패 1회"},
    {"id": "E18", "risk_level": "low", "summary": "점검 중 방화벽 규칙 수정"},
]
# — 여기까지 —

sorted_summaries = [] # 위험도순으로 모을 빈 리스트

# 1. for 로 ["high", "medium", "low"] 를 돌면서 level 에 담으세요
for level in ["high", "medium", "low"]:

    # 2. 그 안에서 for 로 summaries 를 돌면서 s 에 담으세요
    for s in summaries:

        # 3. s 의 risk_level 을 소문자로 바꾼 값이 level 과 같으면 sorted_summaries 에 s 를 append 하세요
        if s["risk_level"].lower() == level:
            sorted_summaries.append(s)

for s in sorted_summaries[0:7]: # 앞의 일곱 건만 출력한다
    print(s["id"], s["risk_level"]) # id 와 위험도를 한 줄에 출력

E01 high
E02 High
E03 high
E04 high
E12 high
E16 HIGH
E05 medium


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-4 · 정렬한 요약을 보고서 줄로 바꾸기</font></h3></td></tr></table>

정렬한 요약을 `- [HIGH] E01 요약` 꼴의 목록 줄로 이어 붙이시오. 위험도는 대문자로 맞춥니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `- [HIGH] E01 admin 계정에 로그인 실패 4회` 로 시작하는 네 줄 |

**💡 힌트**

1. 2교시 문제 2-3 과 같은 모양입니다.
2. `.upper()` 는 `.lower()` 의 반대 — 모두 대문자로 바꿉니다.
3. 보고서에서 위험도가 눈에 띄도록 대문자로 씁니다.

In [20]:
# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
sorted_summaries = [                       # 오전에 위험도순으로 정렬한 결과라고 가정합니다
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 - 탈취 의심"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
]
lines = ""                                 # 목록 줄을 모을 빈 문자열
# — 여기까지 —

# 1. for 로 sorted_summaries 에서 하나씩 꺼내 s 에 담으세요
for s in sorted_summaries:

    # 2. lines 뒤에 f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n" 을 이어 붙이세요
    lines += f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n"

print(lines)                               # 모은 목록 줄을 본다

- [HIGH] E01 admin 계정에 로그인 실패 4회
- [HIGH] E02 실패 직후 admin 로그인 성공 - 탈취 의심
- [MEDIUM] E05 svc_backup 의 sudoers 접근 거부
- [LOW] E06 kim01 로그인 실패 1회



#### 🐍 문법 상자 · 파일을 고치면 왜 다시 읽어야 하나

```python
import importlib
import event_summarizer
importlib.reload(event_summarizer)     # 파일을 다시 읽는다
```

| 상황 | `import` 만 있을 때 | `reload` 를 더하면 |
|---|---|---|
| 처음 실행 | 파일을 읽는다 | 파일을 읽는다 |
| 파일을 고친 뒤 다시 실행 | **처음 읽은 내용을 그대로 쓴다** | 고친 내용을 다시 읽는다 |

노트북은 한 번 `import` 한 파일을 기억해 둡니다. 그래서 확인 셀에 `reload` 줄을 미리 넣어 두었습니다. 파일을 고쳤으면 **문제 셀 → 확인 셀** 순서로 다시 실행합니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-5 · event_summarizer.py 만들기</font></h3></td></tr></table>

오전의 산출물 **`event_summarizer.py`** 를 만드시오. 묶음 요약 함수와 정렬 함수를 파일 하나에 담습니다. 정렬 함수의 빈칸 세 곳을 채웁니다.

1. 아래 셀을 채워 실행하면 파일이 만들어집니다.
2. 그 아래 **확인 셀**을 실행합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 문제 셀은 `Writing event_summarizer.py`, 확인 셀은 `E01 high` · `E02 High` · `E03 high` · `E04 high` · `E05 medium` · `E06 low` |

**💡 힌트**

1. 빈칸 세 곳은 문제 4-3 과 같습니다.
2. `summarize_batch` 와 `summarize_events` 는 3교시에 쓴 줄을 파일에 옮긴 것입니다. 오후에 씁니다.
3. 확인 셀에서 오류가 나면 문제 셀을 고쳐 다시 실행하고, **확인 셀도 다시** 실행합니다.

In [21]:
%%writefile event_summarizer.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

SIZE = 6                                                      # 한 묶음에 넣을 경보 수
# 요약 지시 — 출력 형식(JSON 배열)과 키 이름을 정한다
instruction = "다음 보안 경보 목록을 읽고, 경보마다 id, risk_level, summary 세 키를 가진 JSON 배열로만 답하세요. "
# 위험도 세 단어와 요약 길이를 정한다. 뒤에 경보 목록이 붙는다
instruction = instruction + "risk_level 은 high, medium, low 중 하나입니다. summary 는 한국어 한 문장으로 짧게 씁니다. 경보 목록: "


def summarize_batch(batch):                                   # 묶음 하나를 LLM 으로 요약하는 함수
    # 묶음을 JSON 문자열로 바꿔 지시와 함께 보낸다
    text = call_llm(instruction + json.dumps(batch, ensure_ascii=False))
    result = parse_llm_json(text)                             # 답을 딕셔너리로 바꾼다. 못 읽으면 None
    if result is None:                                        # JSON 으로 읽지 못했으면
        return []                                             # 실패하면 빈 리스트 — 다음 묶음은 계속한다
    return result                                             # 요약 리스트를 돌려준다


def summarize_events(events):                                 # 경보 전체를 묶음으로 나눠 요약하는 함수
    summaries = []                                            # 요약을 모을 빈 리스트
    for start in range(0, len(events), SIZE):                 # 0, 6, 12 … 묶음의 시작 번호
        # 묶음 하나를 요약해 한 건씩 꺼낸다
        for item in summarize_batch(events[start:start + SIZE]):
            summaries.append(item)                            # 요약 하나를 모은다
    return summaries                                          # 모은 요약을 돌려준다


def sort_by_risk(summaries):                                  # 요약을 high → medium → low 순서로 정렬하는 함수
    sorted_summaries = []                                     # 위험도순으로 모을 빈 리스트
    # 1. for 로 ["high", "medium", "low"] 를 돌면서 level 에 담으세요
    for level in ["high", "medium", "low"]:
        # 2. 그 안에서 for 로 summaries 를 돌면서 s 에 담으세요
        for s in summaries:
            # 3. s 의 risk_level 을 소문자로 바꾼 값이 level 과 같으면 sorted_summaries 에 s 를 append 하세요
            if s["risk_level"].lower() == level:
                sorted_summaries.append(s)

    return sorted_summaries                                   # 정렬한 요약을 돌려준다

Writing event_summarizer.py


In [ ]:
# 확인 셀 — 그대로 실행합니다. event_summarizer.py 를 고쳤으면 이 셀을 다시 실행합니다
import importlib                                              # 파일을 다시 읽는 도구
import event_summarizer                                       # 방금 만든 event_summarizer.py 를 불러온다

importlib.reload(event_summarizer)                            # 고친 파일을 다시 읽는다

summaries = [                                                 # 미리 준비한 요약 — 모두 같은 결과를 보도록 고정했다
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 — 탈취 의심"},
    {"id": "E03", "risk_level": "high", "summary": "한 IP 의 비밀번호 스프레잉"},
    {"id": "E04", "risk_level": "high", "summary": "root 계정에 로그인 실패 6회"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
]

for s in event_summarizer.sort_by_risk(summaries):            # 파일 안의 정렬 함수로 정렬한다
    print(s["id"], s["risk_level"])                           # id 와 위험도를 한 줄에 출력

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-6 · 알 수 없는 위험도도 버리지 않기</font></h3></td></tr></table>

`critical` 이나 빈 값처럼 high · medium · low 가 아닌 위험도도 버리지 말고 **맨 뒤에** 붙여 정렬하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `E01 high` · `E05 Medium` · `E06 low` · `E13 critical` · `E16 ` (다섯 줄) |

**💡 힌트**

1. 문제 4-3 의 정렬만 하면 두 건이 사라집니다.
2. `not in ["high", "medium", "low"]` 로 셋 밖의 것을 모읍니다.
3. 리스트 두 개는 `+` 로 이어 붙입니다.

In [22]:
# — 미리 채워 둔 줄입니다. 고치지 않습니다 —
summaries = [
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E13", "risk_level": "critical", "summary": "외부 IP 의 포트 스캔"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
    {"id": "E16", "risk_level": "", "summary": "관리자 권한 계정 무단 생성 의심"},
    {"id": "E05", "risk_level": "Medium", "summary": "svc_backup 의 sudoers 접근 거부"},
]
# — 여기까지 —

# 1. high · medium · low 순서로 모으는 것은 문제 4-3 과 같습니다
sorted_summaries = []
for level in ["high", "medium", "low"]:
    for s in summaries:
        if s["risk_level"].lower() == level:
            sorted_summaries.append(s)

# 2. 셋 가운데 어디에도 들지 않은 요약은 버리지 말고 others 리스트에 모으세요
others = []
for s in summaries:
    if s["risk_level"].lower() not in ["high", "medium", "low"]:
        others.append(s)

# 3. 정렬한 리스트 뒤에 others 를 이어 붙여 sorted_summaries 를 만드세요
sorted_summaries = sorted_summaries + others

# 4. sorted_summaries 의 id 와 risk_level 을 한 줄씩 출력하세요
for s in sorted_summaries:
    print(s["id"], s["risk_level"])

E01 high
E05 Medium
E06 low
E13 critical
E16 


### 📋 4교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 대소문자를 맞춘다 | `s["risk_level"].lower()` |
| 위험도순으로 모은다 | `for level in ["high", "medium", "low"]:` 안에서 `for s in summaries:` |
| 보고서 줄로 쓴다 | `f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n"` |
| 파일로 묶는다 | `event_summarizer.py` — `summarize_events` · `sort_by_risk` |

오전 산출물은 `agent_core/event_summarizer.py` 입니다. 오후에는 이 파일과 `llm_client.py` 로 보고서를 완성합니다.

---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 봅니다</mark>

출력 예상 문제의 답은 문제 바로 아래 ✅ 줄에 있습니다.

### 정답 2-2

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
today = "2026-10-07"                                          # 보고서 날짜
# ── 여기까지 ──

# 1. title 이라는 변수에 f-string 으로 "# 야간 보안 관제 보고 (" + today + ")" 꼴의 제목을 담으세요
title = f"# 야간 보안 관제 보고 ({today})"

print(title)                                                  # 만든 제목을 본다

### 정답 2-3

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1007.json", encoding="utf-8") as f:        # 준비 셀이 만든 파일을 연다
    events = json.load(f)                                     # 경보 18건을 리스트로 읽는다
lines = ""                                                    # 목록 줄을 모을 빈 문자열
# ── 여기까지 ──

# 1. for 로 events 에서 경보를 하나씩 꺼내 event 에 담으세요
for event in events:                                          # 경보를 하나씩 꺼낸다
    # 2. lines 뒤에 f"- [{event['rule']}] {event['id']} {event['detail']}\n" 을 이어 붙이세요
    # 「- [룰] id 내용」 한 줄을 붙인다
    lines = lines + f"- [{event['rule']}] {event['id']} {event['detail']}\n"

print(lines)                                                  # 모은 목록 줄을 본다

### 정답 2-4

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1007.json", encoding="utf-8") as f:         # 준비 셀이 저장한 경보 파일을 연다
    events = json.load(f)                                     # 경보 목록을 리스트로 읽는다
title = "# 야간 보안 관제 보고 (2026-10-07)"
lines = ""                                                    # 목록 줄을 모을 빈 문자열
for event in events:                                          # 경보를 하나씩 꺼낸다
    # 「- [룰] id 내용」 한 줄을 붙인다
    lines = lines + f"- [{event['rule']}] {event['id']} {event['detail']}\n"
# ── 여기까지 ──

# 1. report 라는 변수에 title, "\n\n## 건별 내역\n", lines 를 이어 붙여 담으세요
report = title + "\n\n## 건별 내역\n" + lines

# 2. "report_test.md" 를 쓰기로 열고(encoding="utf-8") report 를 저장하세요 (with 문, f.write)
with open("report_test.md", "w", encoding="utf-8") as f:      # 연습용 보고서 파일을 쓰기로 연다
    f.write(report)                                           # 보고서 문자열을 파일에 쓴다

print("report_test.md 를 저장했습니다. 왼쪽 목록에서 열어 봅니다")

### 정답 2-5

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1007.json", encoding="utf-8") as f:         # 준비 셀이 저장한 경보 파일을 연다
    events = json.load(f)                                     # 경보 목록을 리스트로 읽는다
count = {}                                                    # 룰 이름 → 건수
# ── 여기까지 ──

# 1. for 로 events 에서 경보를 하나씩 꺼내 event 에 담으세요
for event in events:                                          # 경보를 하나씩 꺼낸다
    # 2. rule 이라는 변수에 event 의 rule 값을 담으세요
    rule = event["rule"]                                      # 이 경보의 룰 이름
    # 3. count 의 rule 키 값을 1 늘리세요. 처음 보는 룰이면 1 로 시작합니다 (.get(rule, 0) + 1)
    count[rule] = count.get(rule, 0) + 1                      # 처음 보는 룰이면 0 에서, 아니면 지금 값에서 1 늘린다

for rule in count:                                            # 센 룰 이름을 하나씩 꺼낸다
    print(f"- {rule}: {count[rule]}건")                        # 「- 룰: N건」 꼴로 출력

### 정답 ⭐2-6

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1007.json", encoding="utf-8") as f:         # 준비 셀이 저장한 경보 파일을 연다
    events = json.load(f)                                     # 경보 목록을 리스트로 읽는다
today = "2026-10-07"                                          # 보고서 날짜
# ── 여기까지 ──

# 1. filename 에 f"daily_report_{today}.md" 를 담으세요. 날짜의 - 는 빼고 20261007 꼴로 만듭니다 (.replace)
# 2. 룰 이름마다 "## 룰이름" 소제목을 한 번만 쓰고, 그 아래에 그 룰의 경보를 "- id detail" 로 모으세요
# 3. 만든 보고서를 filename 으로 저장하고, filename 을 출력하세요
filename = f"daily_report_{today.replace('-', '')}.md"        # 날짜의 - 를 빼서 파일 이름을 만든다 — daily_report_20261007.md

report = f"# 야간 보안 관제 보고 ({today})\n"
rules = []                                                    # 룰 이름을 한 번씩만 모을 리스트
for event in events:                                          # 경보를 하나씩 꺼낸다
    if event["rule"] not in rules:                            # 아직 넣지 않은 룰이면
        rules.append(event["rule"])                           # 룰 이름을 넣는다

for rule in rules:                                            # 룰마다 소제목 하나
    report = report + f"\n## {rule}\n"
    for event in events:                                      # 경보를 하나씩 꺼낸다
        if event["rule"] == rule:                             # 이 룰의 경보만
            # 경보 한 줄을 붙인다
            report = report + f"- {event['id']} {event['detail']}\n"

with open(filename, "w", encoding="utf-8") as f:              # 보고서 파일을 쓰기로 연다 (있으면 덮어쓴다)
    f.write(report)                                           # 보고서 문자열을 파일에 쓴다

print(filename)

### 정답 3-2

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1007.json", encoding="utf-8") as f:         # 준비 셀이 저장한 경보 파일을 연다
    events = json.load(f)                                     # 경보 목록을 리스트로 읽는다
# ── 여기까지 ──

# 1. batch 라는 변수에 events 의 0번부터 6개(0:6)를 잘라 담으세요
batch = events[0:6]                                           # 첫 묶음 — 0번부터 5번까지 여섯 건

# 2. batch 의 길이와, 첫 경보와 마지막 경보의 id 를 출력하세요
print(len(batch), batch[0]["id"], batch[-1]["id"])

### 정답 3-3

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1007.json", encoding="utf-8") as f:         # 준비 셀이 저장한 경보 파일을 연다
    events = json.load(f)                                     # 경보 목록을 리스트로 읽는다
size = 6                                                      # 한 묶음에 넣을 경보 수
# ── 여기까지 ──

# 1. for 로 range(0, len(events), size) 를 돌면서 start 에 담으세요
for start in range(0, len(events), size):                     # 0, 6, 12 — 묶음의 시작 번호
    # 2. batch 에 events 의 start 부터 start + size 앞까지 잘라 담으세요
    batch = events[start:start + size]                        # start 부터 여섯 건을 잘라 묶음을 만든다
    # 3. start 와 batch 의 길이와 batch 첫 경보의 id 를 출력하세요
    print(start, len(batch), batch[0]["id"])

### 정답 3-4

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)                                  # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm, parse_llm_json              # 10/6 에 만든 두 함수를 꺼내 쓴다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1007.json", encoding="utf-8") as f:         # 준비 셀이 저장한 경보 파일을 연다
    events = json.load(f)                                     # 경보 목록을 리스트로 읽는다
# 요약 지시 — 출력 형식(JSON 배열)과 키 이름을 정한다
instruction = "다음 보안 경보 목록을 읽고, 경보마다 id, risk_level, summary 세 키를 가진 JSON 배열로만 답하세요. "
# 위험도 세 단어와 요약 길이를 정한다. 뒤에 경보 목록이 붙는다
instruction = instruction + "risk_level 은 high, medium, low 중 하나입니다. summary 는 한국어 한 문장으로 짧게 씁니다. 경보 목록: "
batch = events[0:6]                                           # 첫 묶음 — 0번부터 5번까지 여섯 건
batch_text = json.dumps(batch, ensure_ascii=False)            # 묶음(리스트)을 JSON 문자열로 바꾼다
# ── 여기까지 ──

# 1. text 라는 변수에 call_llm(instruction + batch_text) 의 답을 담으세요
text = call_llm(instruction + batch_text)                     # 지시 + 경보 목록을 LLM 에 보낸다 — 한 번

# 2. result 라는 변수에 parse_llm_json(text) 의 결과를 담으세요
result = parse_llm_json(text)                                 # 답을 딕셔너리로 바꾼다. 못 읽으면 None

# 3. result 의 길이와 첫 요약을 출력하세요
print(len(result))
print(result[0])

### 정답 3-5

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1007.json", encoding="utf-8") as f:         # 준비 셀이 저장한 경보 파일을 연다
    events = json.load(f)                                     # 경보 목록을 리스트로 읽는다
size = 6                                                      # 한 묶음에 넣을 경보 수
summaries = []                                                # 요약을 모을 빈 리스트
# ── 여기까지 ──

# 1. for 로 range(0, len(events), size) 를 돌면서 start 에 담으세요
for start in range(0, len(events), size):                     # 0, 6, 12 — 묶음의 시작 번호
    # 2. batch 에 events[start:start + size] 를 담고, result 에 summarize_batch(batch) 의 결과를 담으세요
    batch = events[start:start + size]                        # start 부터 여섯 건을 잘라 묶음을 만든다
    result = summarize_batch(batch)                           # 이 묶음을 요약한다 — LLM 한 번
    # 3. for 로 result 에서 하나씩 꺼내 summaries 에 append 하세요
    for item in result:                                       # 받은 요약을 하나씩
        summaries.append(item)                                # 요약 하나를 모은다

with open("event_summaries.json", "w", encoding="utf-8") as f:   # 요약을 파일로 남긴다 — 4교시에 다시 쓴다
    json.dump(summaries, f, ensure_ascii=False, indent=2)     # 요약을 파일에 저장한다 — 한글 그대로

print(f"요약 {len(summaries)}건을 event_summaries.json 에 저장했습니다")

### 정답 ⭐3-6

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("events_1007.json", encoding="utf-8") as f:         # 준비 셀이 저장한 경보 파일을 연다
    events = json.load(f)                                     # 경보 목록을 리스트로 읽는다
with open("event_summaries.json", encoding="utf-8") as f:    # 문제 3-5 가 저장한 요약
    summaries = json.load(f)                                  # 저장한 요약을 읽는다
# ── 여기까지 ──

# 1. events 의 id 를 모두 담은 리스트 known_ids 를 만드세요
# 2. summaries 를 돌면서 id 가 known_ids 에 없는 요약을 "[지어낸 id]" 와 함께 출력하세요
# 3. 반대로, events 가운데 요약이 빠진 경보의 id 를 "[빠진 경보]" 와 함께 출력하세요
# 4. 마지막에 "확인 끝" 을 출력하세요
known_ids = []                                                # 원래 경보의 id 를 모을 리스트
for event in events:                                          # 경보를 하나씩 꺼낸다
    known_ids.append(event["id"])                             # 경보 id 를 모은다

summary_ids = []                                              # 요약에 나온 id 를 모을 리스트
for s in summaries:                                           # 요약을 하나씩 꺼낸다
    summary_ids.append(s["id"])                               # 요약 id 를 모은다
    if s["id"] not in known_ids:                              # 원래 경보에 없는 id 면 — 지어낸 것
        print("[지어낸 id]", s["id"])

for event_id in known_ids:                                    # 원래 경보 id 를 하나씩
    if event_id not in summary_ids:                           # 요약에 없는 id 면 — 빠진 것
        print("[빠진 경보]", event_id)

print("확인 끝")

### 정답 4-2

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
summaries = [                                                 # 미리 준비한 요약 — 모두 같은 결과를 보도록 고정했다
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 — 탈취 의심"},
    {"id": "E03", "risk_level": "high", "summary": "한 IP 의 비밀번호 스프레잉"},
    {"id": "E04", "risk_level": "high", "summary": "root 계정에 로그인 실패 6회"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
    {"id": "E07", "risk_level": "low", "summary": "park.js 실패 뒤 정상 로그인"},
    {"id": "E08", "risk_level": "medium", "summary": "deploy 토큰 만료"},
    {"id": "E09", "risk_level": "medium", "summary": "guest 계정에 로그인 실패 5회"},
    {"id": "E10", "risk_level": "medium", "summary": "lee.yh 새 IP 로그인"},
    {"id": "E11", "risk_level": "low", "summary": "choi.mk 로그인 실패 2회"},
    {"id": "E12", "risk_level": "high", "summary": "guest01 의 shadow 접근 거부"},
    {"id": "E13", "risk_level": "Medium", "summary": "외부 IP 의 포트 스캔"},
    {"id": "E14", "risk_level": "low", "summary": "jung.hw 로그인 실패 1회"},
    {"id": "E15", "risk_level": "medium", "summary": "svc_batch 새벽 로그인 실패"},
    {"id": "E16", "risk_level": "HIGH", "summary": "관리자 권한 계정 무단 생성 의심"},
    {"id": "E17", "risk_level": "low", "summary": "kim.cs 로그인 실패 1회"},
    {"id": "E18", "risk_level": "low", "summary": "점검 중 방화벽 규칙 수정"},
]
# ── 여기까지 ──

high_count = 0                                                # high 건수. 0 에서 시작

# 1. for 로 summaries 에서 하나씩 꺼내 s 에 담으세요
for s in summaries:                                           # 요약을 하나씩 꺼낸다
    # 2. s 의 risk_level 을 소문자로 바꾼 값이 "high" 이면 (if 문, .lower())
    if s["risk_level"].lower() == "high":                     # 소문자로 맞춰 high 인지 본다
        # 3. high_count 에 1 을 더하세요
        high_count = high_count + 1                           # high 하나를 센다

print(f"high {high_count}건")

### 정답 4-3

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
summaries = [                                                 # 미리 준비한 요약 — 모두 같은 결과를 보도록 고정했다
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 — 탈취 의심"},
    {"id": "E03", "risk_level": "high", "summary": "한 IP 의 비밀번호 스프레잉"},
    {"id": "E04", "risk_level": "high", "summary": "root 계정에 로그인 실패 6회"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
    {"id": "E07", "risk_level": "low", "summary": "park.js 실패 뒤 정상 로그인"},
    {"id": "E08", "risk_level": "medium", "summary": "deploy 토큰 만료"},
    {"id": "E09", "risk_level": "medium", "summary": "guest 계정에 로그인 실패 5회"},
    {"id": "E10", "risk_level": "medium", "summary": "lee.yh 새 IP 로그인"},
    {"id": "E11", "risk_level": "low", "summary": "choi.mk 로그인 실패 2회"},
    {"id": "E12", "risk_level": "high", "summary": "guest01 의 shadow 접근 거부"},
    {"id": "E13", "risk_level": "Medium", "summary": "외부 IP 의 포트 스캔"},
    {"id": "E14", "risk_level": "low", "summary": "jung.hw 로그인 실패 1회"},
    {"id": "E15", "risk_level": "medium", "summary": "svc_batch 새벽 로그인 실패"},
    {"id": "E16", "risk_level": "HIGH", "summary": "관리자 권한 계정 무단 생성 의심"},
    {"id": "E17", "risk_level": "low", "summary": "kim.cs 로그인 실패 1회"},
    {"id": "E18", "risk_level": "low", "summary": "점검 중 방화벽 규칙 수정"},
]
# ── 여기까지 ──

sorted_summaries = []                                         # 위험도순으로 모을 빈 리스트

# 1. for 로 ["high", "medium", "low"] 를 돌면서 level 에 담으세요
for level in ["high", "medium", "low"]:                       # 이 순서대로 모은다
    # 2. 그 안에서 for 로 summaries 를 돌면서 s 에 담으세요
    for s in summaries:                                       # 요약을 하나씩 꺼낸다
        # 3. s 의 risk_level 을 소문자로 바꾼 값이 level 과 같으면 sorted_summaries 에 s 를 append 하세요
        if s["risk_level"].lower() == level:                  # 소문자로 맞춰 지금 위험도와 같으면
            sorted_summaries.append(s)                        # 정렬 결과에 넣는다

for s in sorted_summaries[0:7]:                               # 앞의 일곱 건만 출력한다
    print(s["id"], s["risk_level"])                           # id 와 위험도를 한 줄에 출력

### 정답 4-4

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
sorted_summaries = [                                          # 오전에 위험도순으로 정렬한 결과라고 가정합니다
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E02", "risk_level": "High", "summary": "실패 직후 admin 로그인 성공 — 탈취 의심"},
    {"id": "E05", "risk_level": "medium", "summary": "svc_backup 의 sudoers 접근 거부"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
]
lines = ""                                                    # 목록 줄을 모을 빈 문자열
# ── 여기까지 ──

# 1. for 로 sorted_summaries 에서 하나씩 꺼내 s 에 담으세요
for s in sorted_summaries:                                    # 정렬된 요약을 하나씩 꺼낸다
    # 2. lines 뒤에 f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n" 을 이어 붙이세요
    # 「- [HIGH] E01 요약」 한 줄을 붙인다
    lines = lines + f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n"

print(lines)                                                  # 모은 목록 줄을 본다

### 정답 4-5

In [ ]:
%%writefile event_summarizer.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

SIZE = 6                                                      # 한 묶음에 넣을 경보 수
# 요약 지시 — 출력 형식(JSON 배열)과 키 이름을 정한다
instruction = "다음 보안 경보 목록을 읽고, 경보마다 id, risk_level, summary 세 키를 가진 JSON 배열로만 답하세요. "
# 위험도 세 단어와 요약 길이를 정한다. 뒤에 경보 목록이 붙는다
instruction = instruction + "risk_level 은 high, medium, low 중 하나입니다. summary 는 한국어 한 문장으로 짧게 씁니다. 경보 목록: "


def summarize_batch(batch):                                   # 묶음 하나를 LLM 으로 요약하는 함수
    # 묶음을 JSON 문자열로 바꿔 지시와 함께 보낸다
    text = call_llm(instruction + json.dumps(batch, ensure_ascii=False))
    result = parse_llm_json(text)                             # 답을 딕셔너리로 바꾼다. 못 읽으면 None
    if result is None:                                        # JSON 으로 읽지 못했으면
        return []                                             # 실패하면 빈 리스트 — 다음 묶음은 계속한다
    return result                                             # 요약 리스트를 돌려준다


def summarize_events(events):                                 # 경보 전체를 묶음으로 나눠 요약하는 함수
    summaries = []                                            # 요약을 모을 빈 리스트
    for start in range(0, len(events), SIZE):                 # 0, 6, 12 … 묶음의 시작 번호
        # 묶음 하나를 요약해 한 건씩 꺼낸다
        for item in summarize_batch(events[start:start + SIZE]):
            summaries.append(item)                            # 요약 하나를 모은다
    return summaries                                          # 모은 요약을 돌려준다


def sort_by_risk(summaries):                                  # 요약을 high → medium → low 순서로 정렬하는 함수
    sorted_summaries = []                                     # 위험도순으로 모을 빈 리스트
    # 1. for 로 ["high", "medium", "low"] 를 돌면서 level 에 담으세요
    for level in ["high", "medium", "low"]:                   # 이 순서대로 모은다
        # 2. 그 안에서 for 로 summaries 를 돌면서 s 에 담으세요
        for s in summaries:                                   # 요약을 하나씩 꺼낸다
            # 3. s 의 risk_level 을 소문자로 바꾼 값이 level 과 같으면 sorted_summaries 에 s 를 append 하세요
            if s["risk_level"].lower() == level:              # 소문자로 맞춰 지금 위험도와 같으면
                sorted_summaries.append(s)                    # 정렬 결과에 넣는다
    return sorted_summaries                                   # 정렬한 요약을 돌려준다

### 정답 ⭐4-6

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
summaries = [                                                 # 미리 준비한 요약 — 모두 같은 결과를 보도록 고정했다
    {"id": "E01", "risk_level": "high", "summary": "admin 계정에 로그인 실패 4회"},
    {"id": "E13", "risk_level": "critical", "summary": "외부 IP 의 포트 스캔"},
    {"id": "E06", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
    {"id": "E16", "risk_level": "", "summary": "관리자 권한 계정 무단 생성 의심"},
    {"id": "E05", "risk_level": "Medium", "summary": "svc_backup 의 sudoers 접근 거부"},
]
# ── 여기까지 ──

# 1. high · medium · low 순서로 모으는 것은 문제 4-3 과 같습니다
# 2. 셋 가운데 어디에도 들지 않은 요약은 버리지 말고 others 리스트에 모으세요
# 3. 정렬한 리스트 뒤에 others 를 이어 붙여 sorted_summaries 를 만드세요
# 4. sorted_summaries 의 id 와 risk_level 을 한 줄씩 출력하세요
sorted_summaries = []                                         # 위험도순으로 모을 빈 리스트
for level in ["high", "medium", "low"]:                       # 이 순서대로 모은다
    for s in summaries:                                       # 요약을 하나씩 꺼낸다
        if s["risk_level"].lower() == level:                  # 소문자로 맞춰 지금 위험도와 같으면
            sorted_summaries.append(s)                        # 정렬 결과에 넣는다

others = []                                                   # 셋 밖의 위험도를 모을 리스트
for s in summaries:                                           # 요약을 하나씩 꺼낸다
    # high · medium · low 가 아니면
    if s["risk_level"].lower() not in ["high", "medium", "low"]:
        others.append(s)                                      # 버리지 않고 모은다

sorted_summaries = sorted_summaries + others                  # 맨 뒤에 이어 붙인다

for s in sorted_summaries:                                    # 정렬된 요약을 하나씩 꺼낸다
    print(s["id"], s["risk_level"])                           # id 와 위험도를 한 줄에 출력